In [1]:
import os
import mlflow
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("1. Initializing Spark & MLflow...")

# MinIO Credentials for Spark & MLflow
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "http://minio:9000"
os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("MINIO_ROOT_USER", "admin")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("MINIO_ROOT_PASSWORD", "password")

spark = SparkSession.builder \
    .appName("NetworkThreat_ML") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"]) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"]) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

# Point MLflow to our internal Docker container
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("Network_Threat_Detection")

print("2. Loading Data from MinIO Delta Lake...")
df = spark.read.format("delta").load("s3a://network-logs/raw-traffic/")

# We will use numerical columns for our first model
assembler = VectorAssembler(
    inputCols=["source_port", "dest_port", "bytes"],
    outputCol="features"
)
data = assembler.transform(df).select("features", col("is_attack").alias("label"))

# Split into training and testing data
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("3. Starting MLflow Run and Training Random Forest...")
with mlflow.start_run(run_name="RF_Run_5"):
    
    # Define the Model Parameters
    num_trees = 10
    max_depth = 5
    
    # Log parameters to MLflow
    mlflow.log_param("num_trees", num_trees)
    mlflow.log_param("max_depth", max_depth)
    
    # Train the Model
    rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=num_trees, maxDepth=max_depth)
    model = rf.fit(train_data)
    
    # Make Predictions
    predictions = model.transform(test_data)
    
    # Evaluate the Model (F1 Score is best for imbalanced attack data)
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
    f1_score = evaluator.evaluate(predictions)
    
    print(f"Model Training Complete! F1 Score: {f1_score:.4f}")
    
    # Log Metric and Model to MLflow
    mlflow.log_metric("f1_score", f1_score)
    mlflow.spark.log_model(model, "random_forest_model")

print("4. Successfully logged to MLflow!")

1. Initializing Spark & MLflow...
2. Loading Data from MinIO Delta Lake...
3. Starting MLflow Run and Training Random Forest...
Model Training Complete! F1 Score: 0.9793
🏃 View run RF_Run_5 at: http://mlflow:5000/#/experiments/1/runs/4b0083e7917e4cb98b6ade45a9ec265c
🧪 View experiment at: http://mlflow:5000/#/experiments/1
4. Successfully logged to MLflow!
